# 評価指標と目的関数

・モデルの予測値の評価に用いるのが評価指標であり、目的関数はモデルが学習中に最適化される関数である。

・上手く学習を進めるには、目的関数が微分可能である必要がある。

## カスタム評価指標とカスタム目的関数
・モデルやライブラリによっては、提供されていない評価指標や目的関数をユーザーが定義できる。

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

# データセットロード
cancer = load_breast_cancer()
X = cancer.data; y = cancer.target
train_x, valid_x, train_y, valid_y = train_test_split(X, y, test_size=0.2, random_state=42)

# xgboostのデータ構造へ変換
dtrain = xgb.DMatrix(train_x, label=train_y)
dvalid = xgb.DMatrix(valid_x, label=valid_y)


# カスタム評価指標: logloss(xgboostの'binary:logisitic'と等価)
def logregobj(preds, dtrain):
    labels = dtrain.get_label()
    preds = 1.0 / (1.0 + np.exp(-preds))
    grad = preds - labels
    hess = preds * (1.0 - preds)
    return grad, hess

# カスタム評価指標: error
def evalerror(preds, dtrain):
    labels = dtrain.get_label()
    return 'custom-error', float(sum(labels != (preds > 0.0))) / len(labels)

# パラメータ
params = {'random_state': 42}
num_round = 50
watchlist = [(dtrain, 'train'), (dvalid, 'eval')]

bst = xgb.train(params, dtrain, num_round, watchlist, obj=logregobj, feval=evalerror, verbose_eval=0)

# 確率値に変換
pred_val = bst.predict(dvalid)
pred = 1.0 / (1.0 + np.exp(-pred_val))
logloss = log_loss(valid_y, pred)
print('logloss: ', logloss)

logloss:  0.09399718567695808


# 評価指標の最適化

## 評価指標最適化のアプローチ

例として一部抜粋

・評価指標がRMSEやloglossならば、モデルの目的関数も同じものを指定できる。RMSLEならば学習データの目的変数を対数変換し、目的関数をRMSEとして学習させた後、指数関数で変換をもとに戻した予測値を提出する。

・異なる評価指標を選択し、それを評価対象としてセットしたEarlyStoppingを用いて、最適になる時点で学習を止める方法。

## 閾値の最適化

予測確率ではなく、正例か負例のラベルを提出する評価指標においては、通常は予測確率からある閾値で分けて、正例か負例に振る。しかしF1-scoreの場
合は正例の割合や正しく予測出来ている割合によって、F1-scoreを最大化する閾値が異なるため、その閾値を求める必要がある。方法として、

・0.01~0.99間を0.01刻みで、すべてのスコアを走査し、最良を探す。

・最適化アルゴリズムを使う。

    ・Nelder-Mead法: 最適化対象となる目的関数が微分可能でなくても使用できる。
    
    ・COBYLA法: 制約式(制約条件)を設定できる。
    
    ・SLSQP法: 最適化対象となる目的関数、制約式が微分可能であることを前提とする。
    
    等。Nelder-MeadやCOBYLAは比較的安定した解が求められる。

In [2]:
from sklearn.metrics import f1_score
from scipy.optimize import minimize

# 0~1まで1万刻みでprob生成
rand = np.random.RandomState(seed=42)
train_y_prob = np.linspace(0, 1, 10000)

# train_y_prob.size分のブール値を真の値として、生成ベクトルを標準化したものを予測値として、それぞれ定義
train_y = pd.Series(rand.uniform(0.0, 1.0, train_y_prob.size) < train_y_prob)
train_pred_prob = np.clip(train_y_prob * np.exp(rand.standard_normal(train_y_prob.shape) * 0.3), 0.0, 1.0)

init_threshold = 0.5
init_score = f1_score(train_y, train_pred_prob >= init_threshold)
print(f'init_threshold: {init_threshold} init_score: {init_score}')


# 目的関数
def f1_opt(x):
    return -f1_score(train_y, train_pred_prob >= x)


# Nelder-Mead法を選択、resultにはベストな閾値が返される
result = minimize(f1_opt, x0=np.array([0.5]), method='Nelder-Mead')
print('\nNelder-Mead method result:\n%s\n' % result)
best_threshold = result['x'].item()
best_score = f1_score(train_y, train_pred_prob >= best_threshold)
print(f'best_threshold: {best_threshold} best_score: {best_score}\n')

# COBYLA法を選択
result = minimize(f1_opt, x0=np.array([0.5]), method='COBYLA', options={'maxiter': 10000, 'catol': 0.8})
print('\nCOBYLA method result:\n%s\n' % result)
best_threshold = result['x'].item()
best_score = f1_score(train_y, train_pred_prob >= best_threshold)
print(f'best_threshold: {best_threshold} best_score: {best_score}')

init_threshold: 0.5 init_score: 0.7221549636803875

Nelder-Mead method result:
 final_simplex: (array([[0.35585937],
       [0.35576172]]), array([-0.76296101, -0.76296101]))
           fun: -0.7629610069536134
       message: 'Optimization terminated successfully.'
          nfev: 31
           nit: 14
        status: 0
       success: True
             x: array([0.35585937])

best_threshold: 0.35585937499999987 best_score: 0.7629610069536134


COBYLA method result:
     fun: -0.7629701400510878
   maxcv: 0.0
 message: 'Optimization terminated successfully.'
    nfev: 22
  status: 1
 success: True
       x: array([0.35644531])

best_threshold: 0.3564453125 best_score: 0.7629701400510878
